# NDT7 (M-Lab) Data Prep — Thailand Broadband + Mobile, Province x Quarter

Aggregates `data/ndt7/th/mlab_th_clean.parquet` (60.2M raw NDT7 test records — 31.0M
broadband, 29.2M cellular, already province-joined) into province x quarter format, split
into Broadband and Mobile/Cellular parts, mirroring `cambodia_ndt7_prep.ipynb` /
`vietnam_ndt7_prep.ipynb`'s combined structure.

This replaces the older ad-hoc Thai NDT7 pipeline (`ndt7_eda.ipynb`, `ndt7_edav2.ipynb`,
`ndt7_mobile_eda.ipynb`, dask-based raw ingestion + BKK-district-specific notebooks) with the
same lean prep-notebook -> province-quarterly-CSV -> combined-EDA-notebook pattern used for
Vietnam and Cambodia, for direct cross-country comparability. Processed in memory-safe
streaming batches (this machine has limited free RAM and 60.2M rows is the largest of the
NDT7 sources) — tile-binning/aggregation formulas are otherwise identical.

**No province-name mapping needed** — Thailand's raw `province` values already match
`data/reference/province_reference.csv` / `data/geo/thailand_provinces.geojson` exactly (77
of 77 verified), unlike Cambodia's Khmer-romanization mismatch.

Same tile scheme as Ookla's own published tiles (zoom-16 slippy tiles, ~610m):
`total_tests >= 100 & n_tiles >= 5`.

**Outputs:**
- `data/exports/ndt7_thailand_province_quarterly.csv` — Broadband
- `data/exports/ndt7_thailand_mobile_province_quarterly.csv` — Mobile/Cellular

In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import warnings
warnings.filterwarnings('ignore')

RAW_PARQUET = '../../../data/ndt7/th/mlab_th_clean.parquet'
TH_REF_CSV = '../../../data/reference/province_reference.csv'

ZOOM = 16
N_TILES = 2 ** ZOOM
MIN_TILE_TESTS = 3
BATCH_SIZE = 1500000

COLS = ['date', 'mean_throughput_mbps', 'min_rtt', 'latitude', 'longitude',
        'type', 'network_type', 'province']

### 1. Streaming Tile-Binning + Partial Aggregation Helpers

In [2]:
PROVINCE_MAP_FN = lambda s: s

def tile_bin_batch(df):
    """Assign zoom-16 tile IDs and quarter labels to a raw batch (no accumulation)."""
    df = df[df['mean_throughput_mbps'] > 0]
    df = df.dropna(subset=['latitude', 'longitude', 'province', 'date'])
    df = df.copy()
    df['province'] = PROVINCE_MAP_FN(df['province'])
    df = df.dropna(subset=['province'])
    if df.empty:
        return df

    df['date'] = pd.to_datetime(df['date'])
    df['year_q'] = (
        df['date'].dt.to_period('Q').astype(str)
        .str.replace(r'(\d{4})Q(\d)', r'\1-Q\2', regex=True)
    )
    df['min_rtt'] = df['min_rtt'].clip(upper=2000)

    lat_rad = np.radians(df['latitude'].clip(-85.05112878, 85.05112878))
    mercator_y = np.log(np.tan(lat_rad) + 1.0 / np.cos(lat_rad))
    df['tile_x'] = ((df['longitude'].astype(float) + 180) / 360 * N_TILES).astype(int).clip(0, N_TILES - 1)
    df['tile_y'] = ((1 - mercator_y / np.pi) / 2 * N_TILES).astype(int).clip(0, N_TILES - 1)
    df['tile_id'] = df['tile_x'].astype(str) + '_' + df['tile_y'].astype(str)
    return df


def partial_tile_agg(df):
    """Per-batch partial sums at (year_q, tile_id, type, network_type) grain — small output,
    safe to accumulate across many batches. sum/count recombine exactly like a single-pass
    groupby would (mean-of-batch-means would NOT be exact; sum/count is)."""
    return df.groupby(['year_q', 'tile_id', 'type', 'network_type']).agg(
        sum_throughput=('mean_throughput_mbps', 'sum'),
        sum_rtt=('min_rtt', 'sum'),
        n=('mean_throughput_mbps', 'count'),
        province=('province', lambda s: s.mode().iat[0]),
    ).reset_index()


def combine_partials(parts):
    """Sum per-batch partials down to one row per (year_q, tile_id, type, network_type)."""
    allp = pd.concat(parts, ignore_index=True)
    combined = allp.groupby(['year_q', 'tile_id', 'type', 'network_type']).agg(
        sum_throughput=('sum_throughput', 'sum'),
        sum_rtt=('sum_rtt', 'sum'),
        n=('n', 'sum'),
        # province is consistent per tile (tiles are ~610m, city-level geolocation almost
        # always unanimous) — take the mode of per-batch modes as a cheap approximation
        province=('province', lambda s: s.mode().iat[0]),
    ).reset_index()
    combined['tile_mean'] = combined['sum_throughput'] / combined['n']
    combined['tile_lat']  = combined['sum_rtt'] / combined['n']
    return combined

### 2. Stream Through Raw Parquet in Batches

60.2M rows — this is the largest NDT7 source, expect this cell to take a while.

In [3]:
pf = pq.ParquetFile(RAW_PARQUET)
print(f"Total rows in file: {pf.metadata.num_rows:,}")

parts = []
rows_seen = 0
for bi, batch in enumerate(pf.iter_batches(columns=COLS, batch_size=BATCH_SIZE)):
    raw_batch = batch.to_pandas()
    rows_seen += len(raw_batch)
    binned = tile_bin_batch(raw_batch)
    if not binned.empty:
        parts.append(partial_tile_agg(binned))
    del raw_batch, binned
    if (bi + 1) % 10 == 0:
        print(f"  processed {rows_seen:,} raw rows so far...")

print(f"Done: {rows_seen:,} raw rows read, {len(parts)} batch-partials to combine")
tile_agg_all = combine_partials(parts)
del parts
tile_agg_all = tile_agg_all[tile_agg_all['n'] >= MIN_TILE_TESTS].copy()
print(f"Tile x quarter x type x network rows after MIN_TILE_TESTS>={MIN_TILE_TESTS} filter: {len(tile_agg_all):,}")
print(tile_agg_all['network_type'].unique())

Total rows in file: 60,248,884


  processed 15,000,000 raw rows so far...


  processed 30,000,000 raw rows so far...


  processed 45,000,000 raw rows so far...


  processed 60,000,000 raw rows so far...


Done: 60,248,884 raw rows read, 41 batch-partials to combine


Tile x quarter x type x network rows after MIN_TILE_TESTS>=3 filter: 15,148
<ArrowStringArray>
['broadband', 'cellular']
Length: 2, dtype: str


### 3. Province-Level Weighted Aggregation (per network type)

In [4]:
def build_province_quarterly(tile_agg_all, network_type, ref):
    tile_agg = tile_agg_all[tile_agg_all['network_type'] == network_type]
    print(f"[{network_type}] tile x quarter x type rows: {len(tile_agg):,}")

    dl = tile_agg[tile_agg['type'] == 'download']
    ul = tile_agg[tile_agg['type'] == 'upload']

    dl_stats = dl.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_d_mbps': np.average(g['tile_mean'], weights=g['n']),
        'avg_lat_ms_wt': np.average(g['tile_lat'], weights=g['n']),
        'total_tests': g['n'].sum(),
        'n_tiles': g['tile_id'].nunique(),
    }), include_groups=False).reset_index()

    ul_stats = ul.groupby(['year_q', 'province']).apply(lambda g: pd.Series({
        'avg_u_mbps': np.average(g['tile_mean'], weights=g['n']),
    }), include_groups=False).reset_index()

    master = pd.merge(dl_stats, ul_stats, on=['year_q', 'province'], how='outer')
    master = master.rename(columns={'year_q': 'quarter'})
    master['year'] = master['quarter'].str.slice(0, 4).astype(int)
    master['quarter.1'] = master['quarter'].str.slice(6, 7).astype(int)

    master['is_reliable'] = (master['total_tests'] >= 100) & (master['n_tiles'] >= 5)
    print(f"[{network_type}] province x quarter rows: {len(master)} | reliable: {master['is_reliable'].sum()} ({master['is_reliable'].mean():.1%})")

    master = master.merge(
        ref[['province_en', 'region', 'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021',
             'density_per_km2', 'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']],
        left_on='province', right_on='province_en', how='left'
    ).drop(columns=['province_en'])

    missing_ref = master[master['region'].isna()]['province'].unique()
    if len(missing_ref):
        print(f"[{network_type}] WARNING — provinces with no reference match: {list(missing_ref)}")

    return master


EXPORT_COLS = ['province', 'quarter', 'year', 'quarter.1', 'avg_d_mbps', 'avg_u_mbps',
               'avg_lat_ms_wt', 'total_tests', 'n_tiles', 'is_reliable', 'region',
               'internet_tier', 'pop_2024', 'gdp_per_capita_raw_2021', 'density_per_km2',
               'gdp_per_capita_usd_ppp_2021', 'gdp_per_capita_thb_2021']

In [5]:
ref = pd.read_csv(TH_REF_CSV)

---
## Part 1 — Broadband

In [6]:
broadband_master = build_province_quarterly(tile_agg_all, 'broadband', ref)
broadband_master.head()

[broadband] tile x quarter x type rows: 11,864


[broadband] province x quarter rows: 907 | reliable: 517 (57.0%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Amnat Charoen,64.426268,162.273822,822.0,5.0,35.014625,2023,1,True,Northeastern,4,372000,77048,113,7939.27,253897.82
1,2023-Q1,Ang Thong,77.277077,114.275989,3236.0,7.0,43.559386,2023,1,True,Central,3,269000,142287,283,14661.70,468881.20
2,2023-Q1,Bangkok Metropolis,98.387089,107.896122,643576.0,49.0,64.833579,2023,1,True,Bangkok & Vicinity,1,5456000,593927,3488,61200.11,1957179.52
3,2023-Q1,Bueng Kan,52.880631,187.636622,373.0,5.0,33.569460,2023,1,True,Northeastern,4,419000,80159,105,8259.84,264149.56
4,2023-Q1,Buri Ram,67.373127,139.465136,3860.0,12.0,42.468914,2023,1,True,Northeastern,4,1566000,80684,155,8313.93,265879.60


In [7]:
out_bb = broadband_master[EXPORT_COLS].copy()
OUT_PATH_BB = '../../../data/exports/ndt7_thailand_province_quarterly.csv'
out_bb.to_csv(OUT_PATH_BB, index=False)
print(f"Exported {len(out_bb)} rows -> {OUT_PATH_BB}")
out_bb.head(3)

Exported 907 rows -> ../../data/exports/ndt7_thailand_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Amnat Charoen,2023-Q1,2023,1,64.426268,35.014625,162.273822,822.0,5.0,True,Northeastern,4,372000,77048,113,7939.27,253897.82
1,Ang Thong,2023-Q1,2023,1,77.277077,43.559386,114.275989,3236.0,7.0,True,Central,3,269000,142287,283,14661.70,468881.20
2,Bangkok Metropolis,2023-Q1,2023,1,98.387089,64.833579,107.896122,643576.0,49.0,True,Bangkok & Vicinity,1,5456000,593927,3488,61200.11,1957179.52


---
## Part 2 — Mobile/Cellular

In [8]:
mobile_master = build_province_quarterly(tile_agg_all, 'cellular', ref)
mobile_master.head()

[cellular] tile x quarter x type rows: 3,284


[cellular] province x quarter rows: 526 | reliable: 81 (15.4%)


,quarter,province,avg_d_mbps,avg_lat_ms_wt,total_tests,n_tiles,avg_u_mbps,year,quarter.1,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,2023-Q1,Ang Thong,8.604534,85.327286,7.0,2.0,3.035360,2023,1,False,Central,3,269000,142287,283,14661.70,468881.20
1,2023-Q1,Bangkok Metropolis,24.603168,86.801761,1316913.0,39.0,12.667808,2023,1,True,Bangkok & Vicinity,1,5456000,593927,3488,61200.11,1957179.52
2,2023-Q1,Buri Ram,12.467345,163.762333,129.0,2.0,12.763042,2023,1,False,Northeastern,4,1566000,80684,155,8313.93,265879.60
3,2023-Q1,Chachoengsao,99.946577,94.527712,278.0,3.0,76.673967,2023,1,False,Eastern,2,733000,400385,142,41256.93,1319396.70
4,2023-Q1,Chai Nat,7.557913,116.196423,71.0,2.0,3.986460,2023,1,False,Central,3,314000,135667,131,13979.56,447066.18


In [9]:
out_mb = mobile_master[EXPORT_COLS].copy()
OUT_PATH_MB = '../../../data/exports/ndt7_thailand_mobile_province_quarterly.csv'
out_mb.to_csv(OUT_PATH_MB, index=False)
print(f"Exported {len(out_mb)} rows -> {OUT_PATH_MB}")
out_mb.head(3)

Exported 526 rows -> ../../data/exports/ndt7_thailand_mobile_province_quarterly.csv


,province,quarter,year,quarter.1,avg_d_mbps,avg_u_mbps,avg_lat_ms_wt,total_tests,n_tiles,is_reliable,region,internet_tier,pop_2024,gdp_per_capita_raw_2021,density_per_km2,gdp_per_capita_usd_ppp_2021,gdp_per_capita_thb_2021
0,Ang Thong,2023-Q1,2023,1,8.604534,3.035360,85.327286,7.0,2.0,False,Central,3,269000,142287,283,14661.70,468881.20
1,Bangkok Metropolis,2023-Q1,2023,1,24.603168,12.667808,86.801761,1316913.0,39.0,True,Bangkok & Vicinity,1,5456000,593927,3488,61200.11,1957179.52
2,Buri Ram,2023-Q1,2023,1,12.467345,12.763042,163.762333,129.0,2.0,False,Northeastern,4,1566000,80684,155,8313.93,265879.60


## Summary

- Input: 60.2M raw NDT7 test records for Thailand (2023–2025): 31.0M broadband, 29.2M
  cellular
- Output: province x quarter aggregates for Broadband and Mobile separately, tile-binned at
  Ookla's zoom-16 resolution, same `is_reliable` threshold as every Ookla country notebook and
  the Vietnam/Cambodia NDT7 prep notebooks
- Processed in streaming batches (memory-safe) — no full 60M-row dataframe ever held in
  memory at once